In [ ]:
spark.sql("SELECT current_catalog()").show()

SparkSession is main entry point for interacting with Apache Spark.

SparkSession lets you execture distributed SQL queries and work with DataFrames

Unity Catalog helps organize and govern data assets in Databricks. 

flowchart TD
    A["Catalog"] --> B["Schema"]
    B --> C["Tables"]
    B --> D["Volumes"]



Volume is a Unity Catalog object used for file based storage

Without IF NOT EXISTS, the second execution could fail because the schema or volume already exists

With it, Databricks creates the object only if it isn't present

This is one aspect of idempotency, an important data engineering principle: repeating an operation should not create unwanted additional effects

However, IF NOT EXISTS does not verify that an existing object has the configuration you intended

In [ ]:
catalog = spark.sql(
    "SELECT current_catalog()"
).first()[0]

schema = "incremental_project"
volume = "source_files"

spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{schema}`
""")

spark.sql(f"""
CREATE VOLUME IF NOT EXISTS
`{catalog}`.`{schema}`.`{volume}`
""")

base_path = f"/Volumes/{catalog}/{schema}/{volume}"

print("Project catalog:", catalog)
print("Project schema:", schema)
print("Volume path:", base_path)


In [ ]:
csv_data = """order_id, customer_id, order_status, order_amount, updated_at
ORD001, C101, delivered, 250.00, 2026-10-01 10:00:00
ORD002, C102, shipped, 150.00, 2026-10-01 10:05:00
ORD003, C103, processing, 450.00, 2026-10-01 10:10:00
ORD004, C104, delivered, 600.00, 2026-10-01 10:15:00
ORD005, C105, shipped, 120.00, 2026-10-01 10:20:00
"""
source_path = f"{base_path}/orders_batch_001.csv"

dbutils.fs.put(source_path, csv_data, overwrite=True)

print("CSV file created at:", source_path)

In [ ]:
display(dbutils.fs.ls(base_path))

In [ ]:
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(source_path)
)

display(df)

In [ ]:
assert df.count() == 5, "Expected exactly 5 records"

assert df.select("order_id").distinct().count() == 5, (
    "Expected 5 unique order IDs"
)

print("SUCCESS: Step 1 validation passed!")